### Q1 - Building the knowledge base

Last two digits of my roll number 1024170424 are 2 and 4.

d = 2 -> category[2 % 3] = category[2] = general
d = 4 -> category[4 % 3] = category[1] = account

so my two personalized entries are one general question and one account question.

In [ ]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

personal_entries = [
    {"question": "what are your contact details", "answer": "You can email us at support@company.com or call our helpline.", "keywords": "contact email helpline", "category": "general"},
    {"question": "how do i update my registered mobile number", "answer": "Go to Profile > Edit Details > Mobile Number to update it.", "keywords": "mobile number update", "category": "account"},
]

faq_df = pd.DataFrame(fixed_entries + personal_entries)
faq_df

### Q2 - Scoring function

For a given query I split it into words and check how many of those words show up in the question, answer and keywords of each entry. Higher count = higher confidence match. Then I sort everything by score.

In [ ]:
def get_matches(query, df):
    query_words = query.lower().split()
    scores = []
    for i in range(len(df)):
        row = df.iloc[i]
        text = (row["question"] + " " + row["answer"] + " " + row["keywords"]).lower()
        count = 0
        for w in query_words:
            if w in text:
                count = count + 1
        scores.append(count)
    result = df.copy()
    result["score"] = scores
    result = result[result["score"] > 0]
    result = result.sort_values(by="score", ascending=False)
    return result

get_matches("how can i pay my fee", faq_df)

### Q3 - Entries by category

Calling it with "account" since that is one of the categories from my personalized entries in Q1.

In [ ]:
def same_category(category_name, df):
    matches = df[df["category"] == category_name]
    return matches["question"]

same_category("account", faq_df)

### Q4 - Add a new keyword

I picked the first entry (annual fee) and I am adding a keyword the user types in.

In [ ]:
new_keyword = input("Enter a new keyword for the annual fee entry: ")
faq_df.loc[0, "keywords"] = faq_df.loc[0, "keywords"] + " " + new_keyword
faq_df.to_csv("1024170424_faq_data.csv", index=False)
faq_df

### Q5 - Count of entries per category

In [ ]:
category_counts = faq_df.groupby("category").size()
category_counts

### Q6 - Handling ties

I changed the scoring function so that after finding the top score, it checks if more than one entry shares that top score. If yes, it prints all of them instead of just picking the first one.

Query "fee" matches both fee related entries with the same score so that produces a tie.
Query "reset password" only matches one entry strongly, so that one does not tie.

In [ ]:
def get_matches_v2(query, df):
    query_words = query.lower().split()
    scores = []
    for i in range(len(df)):
        row = df.iloc[i]
        text = (row["question"] + " " + row["answer"] + " " + row["keywords"]).lower()
        count = 0
        for w in query_words:
            if w in text:
                count = count + 1
        scores.append(count)
    result = df.copy()
    result["score"] = scores
    result = result[result["score"] > 0]
    if len(result) == 0:
        print("no matches found")
        return result
    top_score = result["score"].max()
    tied = result[result["score"] == top_score]
    if len(tied) > 1:
        print("tie found, all top matching entries:")
        print(tied[["question", "score"]])
    else:
        print("single best match:")
        print(tied[["question", "score"]])
    return result.sort_values(by="score", ascending=False)

In [ ]:
get_matches_v2("fee", faq_df)

In [ ]:
get_matches_v2("reset password", faq_df)